In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

<details>
<summary><b>Versioni dei pacchetti</b></summary>

Il codice in questa pagina è stato sviluppato usando i seguenti requisiti.
Si consiglia di utilizzare queste versioni o versioni più recenti.

```
qiskit[all]~=2.3.0
```
</details>

Se hai un insieme di $n$ bit (o qubit), di solito etichetti ciascun bit da $0
\rightarrow n-1$. I diversi software e le diverse risorse devono scegliere come
ordinare questi bit sia nella memoria del computer sia quando vengono visualizzati
sullo schermo.

## Convenzioni di Qiskit

Ecco come l'SDK Qiskit ordina i bit nei diversi scenari.

### Circuiti quantistici

La classe `QuantumCircuit` memorizza i suoi qubit in una lista
(`QuantumCircuit.qubits`). L'indice di un qubit in questa lista definisce
l'etichetta del qubit.

In [1]:
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit import Qubit

qc = QuantumCircuit(2)
qc.qubits[0]  # qubit "0"

Qubit(QuantumRegister(2, "q"), 0)

<Qubit register=(2, "q"), index=0>

### Circuit diagrams

On a circuit diagram, qubit $0$ is the topmost qubit, and qubit $n-1$ the
bottommost qubit. You can change this with the `reverse_bits` argument of
`QuantumCircuit.draw` (see [Change ordering in
Qiskit](#change-ordering-in-qiskit)).

In [2]:
qc.x(1)
qc.draw()

          
q_0: ─────
     ┌───┐
q_1: ┤ X ├
     └───┘

### Diagrammi di circuito
In un diagramma di circuito, il qubit $0$ è il qubit più in alto e il qubit
$n-1$ è quello più in basso. Puoi modificare questo comportamento con l'argomento
`reverse_bits` di `QuantumCircuit.draw` (vedi [Modificare l'ordinamento in
Qiskit](#change-ordering-in-qiskit)).

In [3]:
from qiskit.primitives import StatevectorSampler as Sampler

qc.measure_all()

job = Sampler().run([qc])
result = job.result()
print(f" > Counts: {result[0].data.meas.get_counts()}")

 > Counts: {'10': 1024}


### Strings

When displaying or interpreting a list of bits (or qubits) as a string, bit
$n-1$ is the leftmost bit, and bit $0$ is the rightmost bit. This is because we
usually write numbers with the most significant digit on the left, and in
Qiskit, bit $n-1$ is interpreted as the most significant bit.

For example, the following cell defines a `Statevector` from a string of
single-qubit states. In this case, qubit $0$ is in state $|+\rangle$, and
qubit $1$ in state $|0\rangle$.

In [4]:
from qiskit.quantum_info import Statevector

sv = Statevector.from_label("0+")
sv.probabilities_dict()

{np.str_('00'): np.float64(0.4999999999999999),
 np.str_('01'): np.float64(0.4999999999999999)}

### Interi
Quando si interpretano i bit come un numero, il bit $0$ è il bit meno
significativo e il bit $n-1$ è il più significativo. Questo è utile durante
la programmazione perché ogni bit ha il valore $2^\text{label}$ (dove "label"
è l'indice del qubit in `QuantumCircuit.qubits`). Ad esempio, la seguente
esecuzione di circuito termina con il bit $0$ uguale a `0` e il bit $1$ uguale
a `1`. Questo viene interpretato come il numero intero decimale `2` (misurato
con probabilità `1.0`).

In [5]:
print(sv[1])  # amplitude of state |01>
print(sv[2])  # amplitude of state |10>

(0.7071067811865475+0j)
0j


### Gates

Each gate in Qiskit can interpret a list of qubits in its own way, but
controlled gates usually follow the convention `(control, target)`.

For example, the following cell adds a controlled-X gate where qubit $0$ is
the control and qubit $1$ is the target.

In [6]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
qc.cx(0, 1)
qc.draw()

          
q_0: ──■──
     ┌─┴─┐
q_1: ┤ X ├
     └───┘

### Stringhe
Quando si visualizza o si interpreta una lista di bit (o qubit) come stringa,
il bit $n-1$ è il bit più a sinistra e il bit $0$ è quello più a destra. Questo
accade perché di solito scriviamo i numeri con la cifra più significativa a
sinistra, e in Qiskit il bit $n-1$ è interpretato come il bit più significativo.

Ad esempio, la seguente cella definisce uno `Statevector` a partire da una
stringa di stati a singolo qubit. In questo caso, il qubit $0$ si trova nello
stato $|+\rangle$ e il qubit $1$ nello stato $|0\rangle$.

In [7]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
qc.x(0)
qc.draw(reverse_bits=True)

          
q_1: ─────
     ┌───┐
q_0: ┤ X ├
     └───┘

You can use the `reverse_bits` method to return a new circuit with the
qubits' labels reversed (this does not mutate the original circuit).

In [8]:
qc.reverse_bits().draw()

          
q_0: ─────
     ┌───┐
q_1: ┤ X ├
     └───┘

Questo può causare confusione quando si interpreta una stringa di bit, poiché
ci si potrebbe aspettare che il bit più a sinistra sia il bit $0$, mentre di
solito rappresenta il bit $n-1$.

### Matrici del vettore di stato
Quando si rappresenta uno statevector come lista di numeri complessi (ampiezze),
Qiskit ordina queste ampiezze in modo che l'ampiezza all'indice $x$ rappresenti
lo stato della base computazionale $|x\rangle$.